# ISCE3 and CosiCorr or Micmac Run Notebook

## Get the slc from the RSLC with geocoding by running isce3 processing

In [ ]:
import subprocess
import sys

# Path to yaml
# To do, pass DEM, and RSLC names to a function that updates the yaml automatically
# To do, instead of running offsets or insar, just run GSLC processing workflow
reference_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_20261005T105604_20261005T105616_P05023_N_P_J_001.h5'
secondary_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_UR_RSLC_033_076_A_005_4005_DHDH_A_20261017T105604_20261017T105616_P05023_F_P_J_001.h5'
yaml_file = "/Users/chanagan/Documents/GitHub/ImageryResources/ISCE3_Workflows/NISAR_dense_offsets.yaml"


In [ ]:
cmd = [
    sys.executable,
    "-m",
    "nisar.workflows.insar",
    yaml_file
]

subprocess.run(cmd, check=True)

## Convert slc to tif

In [ ]:
import rasterio
import re 
import os
 
# Path with isce3 files
isce3Path = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/ISCE3'

os.chdir(isce3Path)

files = [
     (
         "./dense_offsets/freqA/HH/reference.slc",
         "./dense_offsets/freqA/HH/reference_amplitude.tif",
     ),
     (
         "./coarse_resample_slc/freqA/HH/coregistered_secondary.slc",
         "./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif",
     ),
 ]
 
for slc_file, out_file in files:
    hdr_file = os.path.splitext(slc_file)[0] + ".hdr"

    with open(hdr_file) as f:
        header = f.read()

    width = int(re.search(r"samples\s*=\s*(\d+)", header, re.I).group(1))
    height = int(re.search(r"lines\s*=\s*(\d+)", header, re.I).group(1))
    dtype_code = int(re.search(r"data type\s*=\s*(\d+)", header, re.I).group(1))

    if dtype_code == 6:
        dtype = np.dtype("<c8")
    elif dtype_code == 4:
        dtype = np.dtype("<f4")
    else:
        raise ValueError(f"Unexpected ENVI data type {dtype_code}: {slc_file}")

    slc = np.memmap(
        slc_file, dtype=dtype, mode="r",
        shape=(height, width),
    )

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "float32",
        "compress": "LZW",
        "zlevel":9,
        "BIGTIFF": "IF_SAFER",
    }

    with rasterio.open(out_file, "w", **profile) as dst:
        for row in range(0, height, 256):
            end = min(row + 256, height)
            amplitude = np.abs(slc[row:end]).astype(np.float32)
            dst.write(amplitude, 1, window=((row, end), (0, width)))

    del slc
    print(f"Saved {out_file}: {width} x {height}")
   

## Correlate with desired correlator

e.g. CosiCorr in docker:

```
# Has to be run from cosicorr github repo folder
docker run --rm -it \
  --platform linux/amd64 \
  --mount "type=bind,source=$PWD,target=/work" \
  -v /Users/chanagan/Downloads:/home/user/Downloads \
  --workdir /work \
  geocosicorr3d:latest \
  bash

python3 scripts/cosicorr.py correlate --step 1 1 ../home/user/Downloads/reference_amplitude.tif \
    ../home/user/Downloads/coregistered_secondary_amplitude.tif 
```

e.g. MicMac in docker

```
# Can be run from folder with files
docker run --rm -it \
  --platform linux/amd64 \
  --mount "type=bind,source=$PWD,target=/work" \
  --workdir /work \
  rupnike/micmac:1.0 \
  bash

mm3d Mm2dPosSism reference_amplitude.tif coregistered_secondary_amplitude.tif \
    Inc=10 Dequant=false DirMEC='MEC_SLCInc10/'
```


In [2]:
import os
import rasterio
import numpy as np

In [ ]:
# Select correlator
# TODO update for micmac
correlator = 'cosicorr' # micmac or cosicorr
work_dir = "/Users/chanagan/Downloads/Panama2026/NISAR_ASF"
displacement_tif = "/Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/reference_amplitude_VS_coregistered_secondary_amplitude_frequency_wz_64_step_8.tif"

if correlator == 'cosicorr':

    corrfactor = 1
    basef = 'CosiCorr'
    output_dir = os.path.dirname(displacement_tif)

    from scipy.ndimage import zoom

    reference_tif = f"{work_dir}/ISCE3/dense_offsets/freqA/HH/reference_amplitude.tif"
    band_names = ["X", "Y", "SNR"]

    with rasterio.open(displacement_tif) as src, \
         rasterio.open(reference_tif) as ref:

        profile = ref.profile.copy()
        profile.update(count=1, dtype='float32', nodata=-9999)

        for band, name in enumerate(band_names, start=1):

            data = src.read(band).astype(np.float32)

            # Upsample to reference dimensions
            output = zoom(
                data,
                (ref.height / src.height, ref.width / src.width),
                order=1
            ).astype(np.float32)

            output_file = os.path.join(output_dir, f"cosicorr_{name}.tif")

            with rasterio.open(output_file, "w", **profile) as dst:
                dst.write(output, 1)

            print(f"Saved: {output_file}")

elif correlator == 'micmac':

    corrfactor = 0.05
    basef = 'micmac'

/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


Saved: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/cosicorr_X.tif
Saved: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/cosicorr_Y.tif
Saved: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/cosicorr_SNR.tif


In [ ]:
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin
from osgeo import gdal
from pyproj import Geod

# --------------------------------------------------
# Configuration
# --------------------------------------------------

rslc_file = (
    f"{work_dir}/RSLC/NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_20261005T105604_20261005T105616_P05023_N_P_J_001.h5"
)

geom_dir = f"{work_dir}/ISCE3/rdr2geo/freqA"

if correlator == 'cosicorr':
    range_file = f"{work_dir}/{basef}/cosicorr_X.tif"
    azimuth_file = f"{work_dir}/{basef}/cosicorr_Y.tif"
elif correlator == 'micmac':
    range_file = f"{work_dir}/{basef}/...tif"
    azimuth_file = f"{work_dir}/{basef}/.._Y.tif"

output_dir = f"{work_dir}/{basef}/geocoded"
os.makedirs(output_dir, exist_ok=True)

# Geolocation subsampling and output resolution
step = 8
resolution = 10
margin = 1000

nodata = -9999.0

# --------------------------------------------------
# Read radar-grid dimensions
# --------------------------------------------------

with rasterio.open(range_file) as src:
    width = src.width
    height = src.height

assert os.path.exists(azimuth_file)
assert os.path.exists(f"{geom_dir}/x.rdr")
assert os.path.exists(f"{geom_dir}/y.rdr")

# --------------------------------------------------
# Derive slant-range spacing from RSLC metadata
# --------------------------------------------------

with h5py.File(rslc_file, "r") as f:
    range_spacing = float(
        f["/science/LSAR/RSLC/swaths/frequencyA/slantRangeSpacing"][()]
    )

# --------------------------------------------------
# Derive ground-projected azimuth spacing from rdr2geo
# --------------------------------------------------

lon_ds = gdal.Open(f"{geom_dir}/x.rdr")
lat_ds = gdal.Open(f"{geom_dir}/y.rdr")

lon_full = lon_ds.ReadAsArray()
lat_full = lat_ds.ReadAsArray()

assert lon_full.shape == (height, width)
assert lat_full.shape == (height, width)

# Subsample the geolocation arrays
lon = lon_full[::500, ::500]
lat = lat_full[::500, ::500]

# Distances between points separated by 500 radar lines
lon1 = lon[:-1, :]
lat1 = lat[:-1, :]
lon2 = lon[1:, :]
lat2 = lat[1:, :]

valid = (
    np.isfinite(lon1) & np.isfinite(lat1) &
    np.isfinite(lon2) & np.isfinite(lat2) &
    (np.abs(lon1) <= 180) & (np.abs(lon2) <= 180) &
    (np.abs(lat1) <= 90) & (np.abs(lat2) <= 90)
)

geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    lon1[valid],
    lat1[valid],
    lon2[valid],
    lat2[valid]
)

distances = distances[np.isfinite(distances) & (distances > 0)]

azimuth_spacing = np.median(distances) / 500.0
print("az spacing std:",np.std(distances/500))

# --------------------------------------------------
# Convert MicMac offsets to displacement in metres
# --------------------------------------------------

range_factor = corrfactor * range_spacing
azimuth_factor = corrfactor * azimuth_spacing

print(f"Slant-range spacing: {range_spacing:.6f} m/pixel")
print(f"Ground-projected azimuth spacing: {azimuth_spacing:.6f} m/pixel")
print(f"Range conversion factor: {range_factor:.6f} m/ unit")
print(f"Azimuth conversion factor: {azimuth_factor:.6f} m/ unit")

def convert_offsets(input_file, output_file, factor):
    with rasterio.open(input_file) as src:
        offsets = src.read(1).astype(np.float32)

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=nodata,
            compress="LZW",
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER"
        )

    valid = np.isfinite(offsets)
    if profile.get("nodata") is not None:
        valid &= offsets != profile["nodata"]

    displacement = np.full(offsets.shape, nodata, dtype=np.float32)
    displacement[valid] = offsets[valid] * factor

    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(displacement, 1)


range_displacement = f"{output_dir}/range_displacement_m.tif"
azimuth_displacement = f"{output_dir}/azimuth_displacement_m.tif"

convert_offsets(
    range_file,
    range_displacement,
    range_factor
)

convert_offsets(
    azimuth_file,
    azimuth_displacement,
    azimuth_factor
)

# --------------------------------------------------
# Create subsampled geolocation arrays
# --------------------------------------------------

lon_geo = lon_full[::step, ::step].astype(np.float32)
lat_geo = lat_full[::step, ::step].astype(np.float32)

geolocation_file = f"{output_dir}/geolocation.tif"

geo_profile = {
    "driver": "GTiff",
    "height": lon_geo.shape[0],
    "width": lon_geo.shape[1],
    "count": 2,
    "dtype": "float32",
    "compress": "LZW",
    "tiled": True,
    "BIGTIFF": "IF_SAFER"
}

with rasterio.open(geolocation_file, "w", **geo_profile) as dst:
    dst.write(lon_geo, 1)
    dst.write(lat_geo, 2)

# --------------------------------------------------
# Geocode images and displacement rasters
# --------------------------------------------------

images = {
    "range_displacement": range_displacement,
    "azimuth_displacement": azimuth_displacement,
}

for name, src_file in images.items():

    src = gdal.Open(src_file)

    src.SetMetadata({
        "X_DATASET": geolocation_file,
        "X_BAND": "1",
        "Y_DATASET": geolocation_file,
        "Y_BAND": "2",
        "SRS": "EPSG:4326",
        "PIXEL_OFFSET": "0",
        "LINE_OFFSET": "0",
        "PIXEL_STEP": str(step),
        "LINE_STEP": str(step),
        "GEOREFERENCING_CONVENTION": "PIXEL_CENTER"
    }, "GEOLOCATION")

    output_file = f"{output_dir}/{name}_utm17.tif"

    warp_options = gdal.WarpOptions(
        format="GTiff",
        geoloc=True,
        dstSRS="EPSG:32617",
        xRes=resolution,
        yRes=resolution,
        targetAlignedPixels=True,
        resampleAlg="bilinear",
        srcNodata=nodata,
        dstNodata=nodata,
        outputBounds=None,
        creationOptions=[
            "COMPRESS=LZW",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    gdal.Warp(output_file, src, options=warp_options)

    src = None
    print(f"Created: {output_file}")

# --------------------------------------------------
# Clean up
# --------------------------------------------------

lon_ds = None
lat_ds = None

print("Done.")

az spacing std: 0.011755063701617323
Slant-range spacing: 3.122838 m/pixel
Ground-projected azimuth spacing: 4.446995 m/pixel
Range conversion factor: 3.122838 m/ unit
Azimuth conversion factor: 4.446995 m/ unit


ERROR 1: missing [


Created: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/geocoded/range_displacement_utm17.tif


ERROR 1: missing [


In [ ]:
# Could replace last 
import os
import h5py
import numpy as np
import rasterio

from osgeo import gdal
from pyproj import Geod
from scipy.ndimage import zoom


# --------------------------------------------------
# Select correlator
# --------------------------------------------------
correlator = 'cosicorr'  # micmac or cosicorr

work_dir = "/Users/chanagan/Downloads/Panama2026/NISAR_ASF"
displacement_tif = (
    f"{work_dir}/CosiCorr/"
    "reference_amplitude_VS_coregistered_secondary_amplitude_frequency_wz_64_step_8.tif"
)

geom_dir = f"{work_dir}/ISCE3/rdr2geo/freqA"

reference_tif = (
    f"{work_dir}/ISCE3/dense_offsets/freqA/HH/reference_amplitude.tif"
)

rslc_file = (
    f"{work_dir}/RSLC/"
    "NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_"
    "20261005T105604_20261005T105616_P05023_N_P_J_001.h5"
)

output_dir = os.path.dirname(displacement_tif)

nodata = -9999.0
output_crs = "EPSG:32617"
output_resolution = 10


# --------------------------------------------------
# Correlator-specific inputs
# --------------------------------------------------
if correlator == 'cosicorr':

    corrfactor = 1.0
    basef = 'CosiCorr'

    with rasterio.open(displacement_tif) as src, \
         rasterio.open(reference_tif) as ref:

        width = ref.width
        height = ref.height

        # Read and upsample X, Y, and SNR bands
        offsets = {}
        source_nodata = src.nodata

        for band, name in enumerate(["X", "Y", "SNR"], start=1):

            data = src.read(band).astype(np.float32)

            # Upsample to reference dimensions
            output = zoom(
                data,
                (height / src.height, width / src.width),
                order=1
            ).astype(np.float32)

            # Ensure exact reference dimensions
            output = output[:height, :width]

            if output.shape != (height, width):
                raise ValueError(
                    f"{name} dimensions {output.shape} do not match "
                    f"reference dimensions {(height, width)}"
                )

            offsets[name] = output

    # COSI-Corr X, Y, and SNR share the same source nodata value
    offset_nodata = {
        "X": source_nodata,
        "Y": source_nodata,
        "SNR": source_nodata
    }


elif correlator == 'micmac':

    corrfactor = 0.05
    basef = 'micmac'

    # Update these paths to your MicMac displacement rasters
    range_file = f"{work_dir}/{basef}/..._X.tif"
    azimuth_file = f"{work_dir}/{basef}/..._Y.tif"

    offsets = {}

    with rasterio.open(range_file) as src:
        offsets["X"] = src.read(1).astype(np.float32)
        range_nodata = src.nodata

    with rasterio.open(azimuth_file) as src:
        offsets["Y"] = src.read(1).astype(np.float32)
        azimuth_nodata = src.nodata

    offset_nodata = {
        "X": range_nodata,
        "Y": azimuth_nodata
    }

    # MicMac SNR output is not included here.


else:
    raise ValueError(f"Unknown correlator: {correlator}")


# --------------------------------------------------
# Read reference dimensions
# --------------------------------------------------
with rasterio.open(reference_tif) as ref:
    width = ref.width
    height = ref.height

print(f"Reference dimensions: {width} x {height}")


# --------------------------------------------------
# Calculate range pixel spacing
# --------------------------------------------------
with h5py.File(rslc_file, "r") as f:

    range_spacing = float(
        f[
            "/science/LSAR/RSLC/swaths/frequencyA/"
            "slantRangeSpacing"
        ][()]
    )

range_factor = corrfactor * range_spacing

print(f"Range spacing: {range_spacing:.4f} m")
print(f"Range conversion factor: {range_factor:.4f}")


# --------------------------------------------------
# Calculate azimuth pixel spacing from geolocation
# --------------------------------------------------
lon_file = f"{geom_dir}/x.rdr"
lat_file = f"{geom_dir}/y.rdr"

lon_ds = gdal.Open(lon_file)
lat_ds = gdal.Open(lat_file)

if lon_ds is None or lat_ds is None:
    raise RuntimeError("Could not open ISCE3 x.rdr or y.rdr")

lon_full = lon_ds.ReadAsArray()
lat_full = lat_ds.ReadAsArray()

if lon_full.shape != (height, width):
    raise ValueError(
        f"Longitude dimensions {lon_full.shape} do not match "
        f"reference dimensions {(height, width)}"
    )

if lat_full.shape != (height, width):
    raise ValueError(
        f"Latitude dimensions {lat_full.shape} do not match "
        f"reference dimensions {(height, width)}"
    )

# Subsample geolocation to estimate azimuth spacing
sample_step = 500

lon = lon_full[::sample_step, ::sample_step]
lat = lat_full[::sample_step, ::sample_step]

lon1, lat1 = lon[:-1, :], lat[:-1, :]
lon2, lat2 = lon[1:, :], lat[1:, :]

valid = (
    np.isfinite(lon1) &
    np.isfinite(lat1) &
    np.isfinite(lon2) &
    np.isfinite(lat2) &
    (np.abs(lon1) <= 180) &
    (np.abs(lon2) <= 180) &
    (np.abs(lat1) <= 90) &
    (np.abs(lat2) <= 90)
)

geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    lon1[valid],
    lat1[valid],
    lon2[valid],
    lat2[valid]
)

distances = distances[
    np.isfinite(distances) & (distances > 0)
]

if distances.size == 0:
    raise RuntimeError("Could not calculate azimuth pixel spacing")

azimuth_spacing = np.median(distances) / sample_step
azimuth_factor = corrfactor * azimuth_spacing

print(f"Azimuth spacing: {azimuth_spacing:.4f} m")
print(f"Azimuth conversion factor: {azimuth_factor:.4f}")

# Release geolocation arrays; GDAL will read the original files
# directly during geocoding.
del lon_full, lat_full, lon, lat
lon_ds = None
lat_ds = None


# --------------------------------------------------
# Geocode displacement and SNR rasters
# --------------------------------------------------
geoloc_metadata = {
    "X_DATASET": lon_file,
    "X_BAND": "1",
    "Y_DATASET": lat_file,
    "Y_BAND": "1",
    "SRS": "EPSG:4326",
    "PIXEL_OFFSET": "0",
    "LINE_OFFSET": "0",
    "PIXEL_STEP": "1",
    "LINE_STEP": "1",
    "GEOREFERENCING_CONVENTION": "PIXEL_CENTER"
}

mem_driver = gdal.GetDriverByName("MEM")

output_names = {
    "X": "range_displacement_utm17.tif",
    "Y": "azimuth_displacement_utm17.tif",
    "SNR": "snr_utm17.tif"
}

conversion_factors = {
    "X": range_factor,
    "Y": azimuth_factor,
    "SNR": 1.0
}

for name, data in offsets.items():

    if data.shape != (height, width):
        raise ValueError(
            f"{name} dimensions {data.shape} do not match "
            f"reference dimensions {(height, width)}"
        )

    # Apply displacement conversion; SNR remains unchanged
    factor = conversion_factors[name]

    valid = np.isfinite(data)

    src_nodata = offset_nodata.get(name)
    if src_nodata is not None:
        valid &= data != src_nodata

    output = np.full(
        (height, width),
        nodata,
        dtype=np.float32
    )

    output[valid] = data[valid] * factor

    # Create an in-memory GDAL raster
    mem = mem_driver.Create(
        "",
        width,
        height,
        1,
        gdal.GDT_Float32
    )

    band = mem.GetRasterBand(1)
    band.WriteArray(output)
    band.SetNoDataValue(nodata)

    # Attach original ISCE3 geolocation arrays
    mem.SetMetadata(geoloc_metadata, "GEOLOCATION")

    # Output filename
    output_file = os.path.join(
        output_dir,
        output_names[name]
    )

    print(f"Geocoding {name}: {output_file}")

    # Geocode directly to the output GeoTIFF
    gdal.Warp(
        output_file,
        mem,
        geoloc=True,
        dstSRS=output_crs,
        xRes=output_resolution,
        yRes=output_resolution,
        targetAlignedPixels=True,
        resampleAlg="nearest",
        srcNodata=nodata,
        dstNodata=nodata,
        creationOptions=[
            "COMPRESS=LZW",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    mem = None

    print(f"Saved: {output_file}")

print("Finished geocoding.")

/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Reference dimensions: 52484 x 19760
Range spacing: 3.1228 m
Range conversion factor: 3.1228


/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Azimuth spacing: 4.4470 m
Azimuth conversion factor: 4.4470
Geocoding X: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/CosiCorr/range_displacement_utm17.tif


ERROR 1: missing [
